# Is the missingness informative?

Every one of the 12 features carries between 4% and 20% missing values, in both train
and test, at similar rates. Real data does not break that way. It looks injected by the
generator, which raises the question of whether it was injected **at random** or
**conditional on something**.

This notebook answers that and nothing else. **It writes no row to the ledger**, because
it trains no model and produces no CV score. It is diagnostic work, and treating a
diagnostic as an experiment is how ledgers fill up with rows that cannot be compared.

Three questions:

1. Does the target rate differ between rows where a feature is present and rows where
   it is missing? If missingness were injected at random, it would not.
2. Does the count of missing fields in a row carry signal?
3. Are the missingness patterns independent across columns, or do they co-occur?

What follows from each is written at the bottom, before the numbers are known, so the
conclusion cannot be fitted to whatever comes out.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

TARGET = "addicted_label"
ID = "id"


def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
BASE = train[TARGET].mean()
print(f"{len(train):,} rows, base rate {BASE:.6f}")

691,369 rows, base rate 0.709424


## 1. Target rate, present versus missing

`lift` is the target rate among rows where the feature is missing, minus the overall
base rate. Under missing-completely-at-random it should sit at zero for every feature,
within sampling error. With 691,369 rows and missing counts in the tens of thousands,
the standard error on these rates is around 0.002, so anything past about 0.006 is
well outside noise.

In [2]:
rows = []
for c in FEATURES:
    m = train[c].isna()
    n_missing = int(m.sum())
    if n_missing == 0:
        continue
    rate_missing = train.loc[m, TARGET].mean()
    rate_present = train.loc[~m, TARGET].mean()
    # standard error of the rate among missing rows
    se = np.sqrt(rate_missing * (1 - rate_missing) / n_missing)
    rows.append({
        "feature": c,
        "pct_missing": 100 * n_missing / len(train),
        "rate_missing": rate_missing,
        "rate_present": rate_present,
        "lift": rate_missing - BASE,
        "z": (rate_missing - BASE) / se,
    })

miss = pd.DataFrame(rows).sort_values("lift", key=abs, ascending=False)
print(f"base rate {BASE:.6f}\n")
print(f"{'feature':<26}{'%miss':>7}{'rate|miss':>11}{'rate|pres':>11}{'lift':>10}{'z':>9}")
print("-" * 74)
for r in miss.itertuples():
    print(f"{r.feature:<26}{r.pct_missing:>7.1f}{r.rate_missing:>11.4f}"
          f"{r.rate_present:>11.4f}{r.lift:>+10.4f}{r.z:>9.1f}")

base rate 0.709424

feature                     %miss  rate|miss  rate|pres      lift        z
--------------------------------------------------------------------------
age                           4.2     0.7134     0.7093   +0.0040      1.5
sleep_hours                   6.4     0.7134     0.7092   +0.0040      1.8
app_opens_per_day            11.7     0.7128     0.7090   +0.0034      2.1
daily_screen_time_hours      13.9     0.7113     0.7091   +0.0019      1.3
weekend_screen_time          16.2     0.7107     0.7092   +0.0012      0.9
work_study_hours              7.5     0.7106     0.7093   +0.0012      0.6
gaming_hours                 18.3     0.7105     0.7092   +0.0011      0.8
notifications_per_day         9.8     0.7102     0.7093   +0.0008      0.5
gender                        4.2     0.7088     0.7095   -0.0006     -0.2
stress_level                  8.0     0.7090     0.7095   -0.0004     -0.2
academic_work_impact          6.4     0.7095     0.7094   +0.0000      0.0
socia

## 2. Does the number of missing fields per row predict the target?

In [3]:
n_miss = train[FEATURES].isna().sum(axis=1)
tab = (pd.DataFrame({"n_missing": n_miss, TARGET: train[TARGET]})
       .groupby("n_missing")[TARGET].agg(["size", "mean"]))
tab["lift"] = tab["mean"] - BASE
print(f"{'n_missing':>10}{'rows':>10}{'rate':>10}{'lift':>10}")
print("-" * 40)
for idx, r in tab.iterrows():
    print(f"{idx:>10}{int(r['size']):>10,}{r['mean']:>10.4f}{r['lift']:>+10.4f}")

corr = np.corrcoef(n_miss, train[TARGET])[0, 1]
print(f"\ncorrelation between missing count and target: {corr:+.4f}")

 n_missing      rows      rate      lift
----------------------------------------
         0   269,185    0.7081   -0.0013
         1   180,459    0.7090   -0.0005
         2   120,697    0.7112   +0.0018
         3    67,328    0.7118   +0.0024
         4    32,557    0.7095   +0.0001
         5    13,678    0.7125   +0.0030
         6     5,157    0.7192   +0.0098
         7     1,676    0.6963   -0.0131
         8       477    0.7023   -0.0071
         9       136    0.6618   -0.0477
        10        18    0.6667   -0.0428
        11         1    1.0000   +0.2906

correlation between missing count and target: +0.0025


## 3. Do the missingness patterns co-occur?

If each column were blanked independently, the pairwise correlations between the
indicator columns would all sit near zero. Structure here would mean the generator
dropped fields in groups, which is itself a feature.

In [4]:
ind = train[FEATURES].isna().astype(int)
cm = ind.corr().to_numpy()
iu = np.triu_indices_from(cm, k=1)
off = cm[iu]
print(f"pairwise correlations between missingness indicators, {len(off)} pairs")
print(f"  min {off.min():+.4f}   max {off.max():+.4f}   mean abs {np.abs(off).mean():.4f}")

k = np.argsort(-np.abs(off))[:5]
print("\nstrongest pairs:")
for j in k:
    a, b = FEATURES[iu[0][j]], FEATURES[iu[1][j]]
    print(f"  {a:<26} {b:<26} {off[j]:+.4f}")

pairwise correlations between missingness indicators, 66 pairs
  min +0.0167   max +0.2756   mean abs 0.0624

strongest pairs:
  social_media_hours         gaming_hours               +0.2756
  notifications_per_day      app_opens_per_day          +0.2596
  social_media_hours         weekend_screen_time        +0.2594
  gaming_hours               weekend_screen_time        +0.2470
  daily_screen_time_hours    social_media_hours         +0.2347


## Train versus test

Missingness rates must be comparable in both. If they are not, that is distribution
shift and it would explain a CV/LB gap far better than any modelling choice.

In [5]:
cmp = pd.DataFrame({
    "train_pct": 100 * train[FEATURES].isna().mean(),
    "test_pct": 100 * test[FEATURES].isna().mean(),
})
cmp["diff"] = cmp["train_pct"] - cmp["test_pct"]
print(f"{'feature':<26}{'train%':>9}{'test%':>9}{'diff':>9}")
print("-" * 53)
for idx, r in cmp.iterrows():
    print(f"{idx:<26}{r['train_pct']:>9.2f}{r['test_pct']:>9.2f}{r['diff']:>+9.2f}")
print(f"\nlargest absolute difference: {cmp['diff'].abs().max():.2f} percentage points")

feature                      train%    test%     diff
-----------------------------------------------------
age                            4.18     5.78    -1.60
daily_screen_time_hours       13.86    11.07    +2.80
social_media_hours            19.38    16.00    +3.38
gaming_hours                  18.34    20.05    -1.71
work_study_hours               7.45     9.37    -1.92
sleep_hours                    6.43     7.58    -1.14
notifications_per_day          9.78    11.55    -1.77
app_opens_per_day             11.67     8.68    +3.00
weekend_screen_time           16.21    17.11    -0.90
gender                         4.20     4.80    -0.60
stress_level                   7.98     6.62    +1.35
academic_work_impact           6.40     8.68    -2.28

largest absolute difference: 3.38 percentage points


## Verdict

Decision rule, fixed before the numbers were seen:

- **Any feature with |lift| past about 0.006** means missingness is not random and
  carries target information. That justifies explicit missing-indicator features as the
  next experiment, since LightGBM's native NaN routing may capture some of this but has
  no way to express "missing here relates to the target in the same direction as missing
  there".
- **A monotone relationship between missing count and the target** justifies a
  row-level missing-count feature, which is a different and cheaper idea than 12
  indicators.
- **Near-zero lift everywhere and a flat missing-count table** means the missingness was
  injected at random, LightGBM already handles it, and this whole line of attack is dead.
  That result gets written into the rejected-ideas list and the effort moves to external
  data.

In [6]:
strong = miss[miss["lift"].abs() > 0.006]
print(f"features with |lift| > 0.006: {len(strong)} of {len(miss)}")
if len(strong):
    print("  " + ", ".join(strong["feature"].tolist()))
    print("\nVERDICT: missingness is informative. Indicator features are worth an experiment.")
else:
    print("\nVERDICT: missingness looks random. Do not build indicator features. "
          "Log as rejected and move to external data.")
print(f"\nmissing-count correlation with target: {corr:+.4f}")
print(f"max train/test missingness difference: {cmp['diff'].abs().max():.2f} pp")

features with |lift| > 0.006: 0 of 12

VERDICT: missingness looks random. Do not build indicator features. Log as rejected and move to external data.

missing-count correlation with target: +0.0025
max train/test missingness difference: 3.38 pp
